In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import re
import html
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from langdetect import detect, DetectorFactory, LangDetectException
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# --- CORE PROCESSING ---

def parse_webnlg_multilingual(root_path):
    """Parses the specific folder structure and extracts aligned lexicalisations."""
    data = []
    root = Path(root_path)
    
    for xml_file in root.rglob("*.xml"):
        tree = ET.parse(xml_file)
        for entry in tree.findall(".//entry"):
            eid = entry.get("eid")
            category = entry.get("category")
            # Get English Gold Triples for grounding
            triples_en = [t.text for t in entry.findall(".//originaltripleset/otriple")]
            
            # Group lexicalisations by lid to align EN-Gold with Target-Silver
            lex_map = {}
            for lex in entry.findall("lex"):
                lang = lex.get("lang")
                lid = lex.get("lid")
                text = lex.text
                if lang not in lex_map: lex_map[lang] = {}
                lex_map[lang][lid] = text
            
            # Create pairs for each target language
            if 'en' in lex_map:
                for target_lang in ['es', 'ca']:
                    if target_lang in lex_map:
                        for lid, gold_text in lex_map['en'].items():
                            if lid in lex_map[target_lang]:
                                data.append({
                                    'eid': eid,
                                    'category': category,
                                    'lid': lid,
                                    'lang': target_lang,
                                    'text_en': gold_text,
                                    'text_target': lex_map[target_lang][lid],
                                    'triples_en': triples_en
                                })
    return pd.DataFrame(data)

In [2]:
# Ensure deterministic language detection
DetectorFactory.seed = 42

# --- UTILS ---
def basic_norm(s):
    if not s: return ""
    return html.unescape(s).strip()

def norm_for_match(s):
    s = basic_norm(s).lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.replace("_", " ")
    s = re.sub(r"[\(\)\[\]\{\},;:!?\"']", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def extract_digits(text):
    return re.findall(r"\d+(?:[.,]\d+)?", text or "")

def get_entity_tokens(triples):
    tokens = set()
    for tr in triples:
        parts = [p.strip() for p in tr.split('|')]
        for p in [parts[0], parts[-1]]:
            norm_p = norm_for_match(p)
            tokens.update([t for t in norm_p.split() if len(t) > 2])
    return tokens

def perform_deep_comparative_analysis(df_results):
    """
    Computes metrics for Full, Under-Knee, and Over-Knee segments.
    """
    final_report = []
    metrics = [
        'semantic_alignment', 
        'triple_realization', 
        'digit_preservation', 
        'expansion_ratio'
    ]

    for lang in df_results['lang'].unique():
        ldf = df_results[df_results['lang'] == lang].copy()
        
        # 1. Compute Knee Threshold for this language
        thresh = find_knee_point(ldf['semantic_alignment'].values)
        
        # 2. Segment the data
        under_knee = ldf[ldf['semantic_alignment'] < thresh]
        over_knee = ldf[ldf['semantic_alignment'] >= thresh]
        
        # 3. Build the stats dictionary
        # We use a nested approach to later pivot/format for the paper
        lang_stats = {
            'Language': lang,
            'Knee_Threshold': round(thresh, 4),
            'Tail_Size_%': round(len(under_knee) / len(ldf) * 100, 2),
            'Full_Count': len(ldf)
        }

        for m in metrics:
            # Calculate means for the three segments
            mean_full = ldf[m].mean()
            mean_under = under_knee[m].mean()
            mean_over = over_knee[m].mean()
            
            lang_stats[f'{m}_Full'] = round(mean_full, 4)
            lang_stats[f'{m}_Under'] = round(mean_under, 4)
            lang_stats[f'{m}_Over'] = round(mean_over, 4)
            # The "Improvement" or "Gap" by filtering
            lang_stats[f'{m}_Delta'] = round(mean_over - mean_under, 4)

        final_report.append(lang_stats)
    return pd.DataFrame(final_report)

# --- MEMORY EFFICIENT EVALUATOR ---

class WebNLGEvaluator:
    def __init__(self, model_name="intfloat/multilingual-e5-base", batch_size=32):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading model on {self.device}...")
        self.model = SentenceTransformer(model_name, device=self.device)
        self.batch_size = batch_size

    def compute_semantic_alignment(self, en_texts, tgt_texts):
        """Computes cosine similarity in batches to avoid OOM."""
        all_sims = []
        
        # Add E5 specific prefixes
        en_inputs = [f"query: {t}" for t in en_texts]
        tgt_inputs = [f"passage: {t}" for t in tgt_texts]

        for i in tqdm(range(0, len(en_inputs), self.batch_size), desc="Embedding Batches"):
            batch_en = en_inputs[i:i + self.batch_size]
            batch_tgt = tgt_inputs[i:i + self.batch_size]
            
            # Encode to tensors
            emb_en = self.model.encode(batch_en, convert_to_tensor=True, show_progress_bar=False)
            emb_tgt = self.model.encode(batch_tgt, convert_to_tensor=True, show_progress_bar=False)
            
            # Row-wise Cosine Similarity: Dot product of normalized vectors
            # This avoids creating the N x N matrix that causes OOM
            emb_en = torch.nn.functional.normalize(emb_en, p=2, dim=1)
            emb_tgt = torch.nn.functional.normalize(emb_tgt, p=2, dim=1)
            
            sims = (emb_en * emb_tgt).sum(dim=1)
            all_sims.extend(sims.cpu().tolist())
            
            # Clear cache for safety
            del emb_en, emb_tgt
            if self.device == "cuda": torch.cuda.empty_cache()
            
        return all_sims

    def evaluate(self, df):
        print(f"Processing {len(df)} rows...")
        
        # 1. RQ2: Semantic Alignment
        df['semantic_alignment'] = self.compute_semantic_alignment(
            df['text_en'].tolist(), 
            df['text_target'].tolist()
        )

        # 2. RQ1: Faithfulness & Validity
        results = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Calculating RQ1"):
            # Lang Consistency
            try:
                det = detect(row['text_target'])
                lang_score = 1.0 if det == row['lang'] else 0.0
            except: lang_score = 0.0
            
            # Expansion Ratio
            en_len = len(row['text_en'].split())
            tgt_len = len(row['text_target'].split())
            exp_ratio = tgt_len / en_len if en_len > 0 else 1.0
            
            # Triple Realization & Digits
            en_entities = get_entity_tokens(row['triples_en'])
            norm_tgt = norm_for_match(row['text_target'])
            realized = [t for t in en_entities if t in norm_tgt]
            triple_score = 1.0 if not en_entities else len(realized) / len(en_entities)
            
            en_digits = set(extract_digits(" ".join(row['triples_en'])))
            tgt_digits = set(extract_digits(row['text_target']))
            digit_score = 1.0 if not en_digits else len(en_digits & tgt_digits) / len(en_digits)

            results.append({
                'lang_consistency': lang_score,
                'expansion_ratio': exp_ratio,
                'digit_preservation': digit_score,
                'triple_realization': triple_score
            })
        
        return pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)

# --- KNEE DETECTION & ANALYSIS ---

def find_knee_point(values):
    if len(values) < 3: return np.mean(values) if len(values) > 0 else 0
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    coords = np.column_stack((np.linspace(0, 1, n), sorted_vals))
    line_vec = coords[-1] - coords[0]
    line_vec_norm = line_vec / np.linalg.norm(line_vec)
    vec_from_start = coords - coords[0]
    scalar_proj = np.dot(vec_from_start, line_vec_norm)
    vec_proj = np.outer(scalar_proj, line_vec_norm)
    dist_to_line = np.linalg.norm(vec_from_start - vec_proj, axis=1)
    return sorted_vals[np.argmax(dist_to_line)]

# --- MAIN EXECUTION ---

df = parse_webnlg_multilingual("WebNLG_CA_BT")

# Assuming 'df' is your parsed DataFrame
# If it's very large, process one language at a time
evaluator = WebNLGEvaluator(batch_size=16) # Lower batch size if still OOM
df_results = evaluator.evaluate(df)

report_data = []
for lang in df_results['lang'].unique():
    ldf = df_results[df_results['lang'] == lang].copy()
    thresh = find_knee_point(ldf['semantic_alignment'].values)
    
    under = ldf[ldf['semantic_alignment'] < thresh]
    over = ldf[ldf['semantic_alignment'] >= thresh]
    
    res = {'Lang': lang, 'Knee': round(thresh, 3), 'Tail_%': round(len(under)/len(ldf)*100, 1)}
    for m in ['semantic_alignment', 'triple_realization', 'digit_preservation']:
        res[f'{m}_Aligned'] = round(over[m].mean(), 3)
        res[f'{m}_Gap'] = round(over[m].mean() - under[m].mean(), 3)
    report_data.append(res)

final_comparison = pd.DataFrame(report_data)
print(final_comparison.T)

Loading model on cuda...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing 103052 rows...


Embedding Batches:   0%|          | 0/6441 [00:00<?, ?it/s]

Calculating RQ1:   0%|          | 0/103052 [00:00<?, ?it/s]

                                0      1
Lang                           es     ca
Knee                        0.849   0.86
Tail_%                        6.4    7.2
semantic_alignment_Aligned  0.894  0.896
semantic_alignment_Gap       0.06  0.048
triple_realization_Aligned  0.615   0.62
triple_realization_Gap      0.167  0.097
digit_preservation_Aligned  0.843  0.832
digit_preservation_Gap      -0.05 -0.077
